In [8]:
import pandas as pd 
import numpy as np 


In [9]:
# =====================
# Load base files
# =====================

patient = pd.read_csv('data_clinical_patient.txt', sep='\t', comment='#')
sample = pd.read_csv('data_clinical_sample.txt', sep='\t', comment='#')
treatment = pd.read_csv('data_timeline_treatment.txt', sep='\t', comment='#')
progression = pd.read_csv('data_timeline_progression.txt', sep='\t', comment='#')
pdl1 = pd.read_csv('data_timeline_pdl1.txt', sep='\t', comment='#')


In [10]:
# =====================
# Step 1: Merge patient + sample (one row per patient)
# =====================

df = pd.merge(patient, sample, on='PATIENT_ID', how='inner')
df_nsclc = df[df['CANCER_TYPE'].str.contains('Non-Small', na=False)].copy()
print(f"NSCLC patients: {len(df_nsclc)}")


NSCLC patients: 7809


In [11]:
# =====================
# Step 2: Aggregate treatment per patient
# Each patient gets one row with all their treatments
# =====================
print("\nTreatment columns:", treatment.columns.tolist())
print(treatment.head(3))

# Get unique treatment agents per patient
treatment_agg = treatment.groupby('PATIENT_ID').agg(
    TREATMENT_COUNT=('AGENT', 'count'),
    TREATMENT_AGENTS=('AGENT', lambda x: '|'.join(x.dropna().unique())),
    TREATMENT_SUBTYPES=('SUBTYPE', lambda x: '|'.join(x.dropna().unique())),
    FIRST_TREATMENT_START=('START_DATE', 'min'),
    LAST_TREATMENT_STOP=('STOP_DATE', 'max')
).reset_index()

print(f"\nTreatment aggregated shape: {treatment_agg.shape}")
print(treatment_agg.head(3))



Treatment columns: ['PATIENT_ID', 'START_DATE', 'STOP_DATE', 'EVENT_TYPE', 'SUBTYPE', 'AGENT', 'RX_INVESTIGATIVE', 'FLAG_OROTOPICAL']
  PATIENT_ID  START_DATE  STOP_DATE EVENT_TYPE SUBTYPE             AGENT  \
0  P-0000012       -5437      -5369  Treatment   Chemo  CYCLOPHOSPHAMIDE   
1  P-0000012       -5437      -5326  Treatment   Chemo      FLUOROURACIL   
2  P-0000012       -5437      -5327  Treatment   Chemo      METHOTREXATE   

  RX_INVESTIGATIVE  FLAG_OROTOPICAL  
0                N                0  
1                N                0  
2                N                0  

Treatment aggregated shape: (21473, 6)
  PATIENT_ID  TREATMENT_COUNT  \
0  P-0000012                9   
1  P-0000015               18   
2  P-0000036                5   

                                    TREATMENT_AGENTS  \
0  CYCLOPHOSPHAMIDE|FLUOROURACIL|METHOTREXATE|CIS...   
1  TAMOXIFEN|FULVESTRANT|INVESTIGATIONAL|CAPECITA...   
2      BEVACIZUMAB|CARBOPLATIN|PEMETREXED|CRIZOTINIB   

          

In [12]:
# =====================
# Step 3: Aggregate progression per patient
# =====================
print("\nProgression columns:", progression.columns.tolist())

progression_agg = progression.groupby('PATIENT_ID').agg(
    PROGRESSION_COUNT=('PROGRESSION', 'count'),
    HAS_PROGRESSION=('PROGRESSION', lambda x: 1 if 'Y' in x.values else 0),
    NLP_PROGRESSION_MAX=('NLP_PROGRESSION_PROBABILITY', 'max')
).reset_index()

print(f"\nProgression aggregated shape: {progression_agg.shape}")



Progression columns: ['PATIENT_ID', 'START_DATE', 'STOP_DATE', 'EVENT_TYPE', 'SUBTYPE', 'PROGRESSION', 'STYLE_COLOR', 'NLP_PROGRESSION_PROBABILITY', 'PROCEDURE_TYPE']

Progression aggregated shape: (24370, 4)


In [13]:
# =====================
# Step 4: Aggregate PDL1 per patient
# =====================
print("\nPDL1 columns:", pdl1.columns.tolist())

pdl1_agg = pdl1.groupby('PATIENT_ID').agg(
    PDL1_POSITIVE_ANY=('PDL1_POSITIVE', lambda x: 1 if 'Positive' in x.values else 0)
).reset_index()

print(f"\nPDL1 aggregated shape: {pdl1_agg.shape}")



PDL1 columns: ['PATIENT_ID', 'START_DATE', 'STOP_DATE', 'EVENT_TYPE', 'SUBTYPE', 'SOURCE', 'PDL1_POSITIVE']

PDL1 aggregated shape: (4861, 2)


In [14]:
# =====================
# Step 5: Merge everything (one row per patient)
# =====================
df_final = df_nsclc.copy()
df_final = pd.merge(df_final, treatment_agg, on='PATIENT_ID', how='left')
df_final = pd.merge(df_final, progression_agg, on='PATIENT_ID', how='left')
df_final = pd.merge(df_final, pdl1_agg, on='PATIENT_ID', how='left')

print(f"\nFinal shape: {df_final.shape}")
print(f"Expected: ~7809 rows")
print(f"\nSample:")
print(df_final[['PATIENT_ID', 'OS_MONTHS', 'OS_STATUS', 
                 'TREATMENT_AGENTS', 'HAS_PROGRESSION', 
                 'PDL1_POSITIVE_ANY']].head(10))

# Save
df_final.to_csv('nsclc_merged_clean.csv', index=False)
print("\nFile saved: nsclc_merged_clean.csv")


Final shape: (7809, 58)
Expected: ~7809 rows

Sample:
  PATIENT_ID   OS_MONTHS   OS_STATUS  \
0  P-0000012  118.454665    0:LIVING   
1  P-0000036  115.462887    0:LIVING   
2  P-0000082  116.219051    0:LIVING   
3  P-0000110   47.934194  1:DECEASED   
4  P-0000133   19.594499  1:DECEASED   
5  P-0000113   32.515033    0:LIVING   
6  P-0000165   15.682175  1:DECEASED   
7  P-0000168   82.520458    0:LIVING   
8  P-0000149   18.739705  1:DECEASED   
9  P-0000163    0.032877  1:DECEASED   

                                    TREATMENT_AGENTS  HAS_PROGRESSION  \
0  CYCLOPHOSPHAMIDE|FLUOROURACIL|METHOTREXATE|CIS...              1.0   
1      BEVACIZUMAB|CARBOPLATIN|PEMETREXED|CRIZOTINIB              1.0   
2  BEVACIZUMAB|PEMETREXED|DOCETAXEL|INVESTIGATION...              1.0   
3  BEVACIZUMAB|PACLITAXEL|PEMETREXED|ZOLEDRONIC ACID              1.0   
4                  LETROZOLE|GEMCITABINE|VINORELBINE              1.0   
5         ETOPOSIDE|CISPLATIN|CARBOPLATIN|DURVALUMAB              

In [16]:
df_final.head()

,PATIENT_ID,GENDER,RACE,ETHNICITY,CURRENT_AGE_DEID,STAGE_HIGHEST_RECORDED,NUM_ICDO_DX,ADRENAL_GLANDS,BONE,CNS_BRAIN,...,TMB_NONSYNONYMOUS,TREATMENT_COUNT,TREATMENT_AGENTS,TREATMENT_SUBTYPES,FIRST_TREATMENT_START,LAST_TREATMENT_STOP,PROGRESSION_COUNT,HAS_PROGRESSION,NLP_PROGRESSION_MAX,PDL1_POSITIVE_ANY
0,P-0000012,Female,White,Non-Spanish; Non-Hispanic,68.0,Stage 1-3,2,No,No,No,...,32.165504,9.0,CYCLOPHOSPHAMIDE|FLUOROURACIL|METHOTREXATE|CIS...,Chemo|Investigational|Immuno,-5437.0,1814.0,44.0,1.0,NaN,NaN
1,P-0000036,Female,Other,Non-Spanish; Non-Hispanic,68.0,Stage 4,1,No,Yes,No,...,7.764087,5.0,BEVACIZUMAB|CARBOPLATIN|PEMETREXED|CRIZOTINIB,Biologic|Chemo|Targeted,-199.0,3512.0,50.0,1.0,NaN,NaN
2,P-0000082,Male,White,Non-Spanish; Non-Hispanic,70.0,Stage 1-3,1,No,Yes,No,...,13.309864,9.0,BEVACIZUMAB|PEMETREXED|DOCETAXEL|INVESTIGATION...,Biologic|Chemo|Investigational|Hormone,-1056.0,1023.0,66.0,1.0,NaN,NaN
3,P-0000110,Male,White,Non-Spanish; Non-Hispanic,75.0,Stage 4,1,No,Yes,Yes,...,16.637330,4.0,BEVACIZUMAB|PACLITAXEL|PEMETREXED|ZOLEDRONIC ACID,Biologic|Chemo|Bone Treatment,-1637.0,-853.0,39.0,1.0,NaN,NaN
4,P-0000133,Female,White,Non-Spanish; Non-Hispanic,83.0,Stage 1-3,3,Yes,Yes,No,...,4.436621,4.0,LETROZOLE|GEMCITABINE|VINORELBINE,Hormone|Chemo,-807.0,145.0,21.0,1.0,NaN,NaN


In [17]:
import pandas as pd

df = pd.read_csv('nsclc_merged_clean.csv')

# =====================
# Check 1: Treatment distribution
# =====================
print("=== Treatment Subtype Distribution ===")
print(df['TREATMENT_SUBTYPES'].value_counts().head(20))

# =====================
# Check 2: OS Status distribution
# =====================
print("\n=== OS Status ===")
print(df['OS_STATUS'].value_counts())

# =====================
# Check 3: Missing values in key columns
# =====================
key_cols = ['OS_MONTHS', 'OS_STATUS', 'TREATMENT_AGENTS', 
            'TREATMENT_SUBTYPES', 'PDL1_POSITIVE_ANY',
            'HAS_PROGRESSION', 'STAGE_HIGHEST_RECORDED',
            'SMOKING_PREDICTIONS_3_CLASSES']

print("\n=== Missing Values in Key Columns ===")
for col in key_cols:
    if col in df.columns:
        missing = df[col].isna().sum()
        pct = missing / len(df) * 100
        print(f"{col:40} {missing:5} ({pct:.1f}%)")

# =====================
# Check 4: Categorize treatments
# =====================
def categorize_treatment(agents):
    # Define treatment categories based on drug names
    if pd.isna(agents):
        return 'Unknown'
    
    agents_upper = agents.upper()
    
    immuno_drugs = ['PEMBROLIZUMAB', 'NIVOLUMAB', 'ATEZOLIZUMAB',
                    'DURVALUMAB', 'IPILIMUMAB', 'CEMIPLIMAB']
    targeted_drugs = ['ERLOTINIB', 'GEFITINIB', 'OSIMERTINIB',
                      'CRIZOTINIB', 'ALECTINIB', 'LORLATINIB',
                      'AFATINIB', 'DACOMITINIB']
    
    has_immuno = any(drug in agents_upper for drug in immuno_drugs)
    has_targeted = any(drug in agents_upper for drug in targeted_drugs)
    
    if has_immuno and has_targeted:
        return 'Immuno+Targeted'
    elif has_immuno:
        return 'Immunotherapy'
    elif has_targeted:
        return 'Targeted'
    else:
        return 'Chemotherapy'

df['TREATMENT_CATEGORY'] = df['TREATMENT_AGENTS'].apply(
    categorize_treatment)

print("\n=== Treatment Categories ===")
print(df['TREATMENT_CATEGORY'].value_counts())

# Save updated file
df.to_csv('nsclc_merged_clean.csv', index=False)
print("\nFile updated and saved!")

=== Treatment Subtype Distribution ===
TREATMENT_SUBTYPES
Chemo                          919
Chemo|Immuno                   909
Targeted                       399
Immuno                         196
Chemo|Targeted                 186
Chemo|Immuno|Biologic          138
Investigational                128
Hormone                        122
Targeted|Investigational       109
Biologic|Chemo                  93
Immuno|Chemo                    83
Chemo|Biologic                  77
Chemo|Immuno|Targeted           76
Chemo|Hormone                   75
Biologic|Chemo|Immuno           75
Chemo|Investigational           74
Targeted|Chemo                  73
Chemo|Immuno|Bone Treatment     61
Chemo|Biologic|Immuno           58
Targeted|Biologic|Chemo         57
Name: count, dtype: int64

=== OS Status ===
OS_STATUS
1:DECEASED    3975
0:LIVING      3834
Name: count, dtype: int64

=== Missing Values in Key Columns ===
OS_MONTHS                                    0 (0.0%)
OS_STATUS                     

In [18]:
import pandas as pd
import numpy as np

# Load cleaned data
df = pd.read_csv('nsclc_merged_clean.csv')

# =====================
# Step 1: Simplify treatment categories
# =====================
def simplify_treatment(cat):
    # Merge Immuno+Targeted into Immunotherapy
    if cat == 'Immuno+Targeted':
        return 'Immunotherapy'
    return cat

df['TREATMENT_FINAL'] = df['TREATMENT_CATEGORY'].apply(
    simplify_treatment)

# Remove unknown treatment patients
df = df[df['TREATMENT_FINAL'] != 'Unknown'].copy()
print(f"Patients after removing unknown treatment: {len(df)}")

# =====================
# Step 2: Encode OS_STATUS
# =====================
df['OS_EVENT'] = df['OS_STATUS'].apply(
    lambda x: 1 if '1:DECEASED' in str(x) else 0)

# =====================
# Step 3: Encode GENDER
# =====================
df['GENDER_ENC'] = df['GENDER'].map(
    {'Male': 1, 'Female': 0})

# =====================
# Step 4: Encode SMOKING
# =====================
print("\n=== Smoking Values ===")
print(df['SMOKING_PREDICTIONS_3_CLASSES'].value_counts())

# =====================
# Step 5: Encode STAGE
# =====================
print("\n=== Stage Values ===")
print(df['STAGE_HIGHEST_RECORDED'].value_counts())

# =====================
# Step 6: Encode CANCER_TYPE_DETAILED
# =====================
print("\n=== Cancer Subtypes ===")
print(df['CANCER_TYPE_DETAILED'].value_counts().head(10))

# =====================
# Step 7: Check TMB and MSI
# =====================
print("\n=== TMB Stats ===")
print(df['TMB_NONSYNONYMOUS'].describe())

print("\n=== MSI Stats ===")
print(df['MSI_SCORE'].describe())

# Save
df.to_csv('nsclc_step1.csv', index=False)
print("\nStep 1 saved!")

Patients after removing unknown treatment: 6111

=== Smoking Values ===
SMOKING_PREDICTIONS_3_CLASSES
Former/Current Smoker    4180
Never                    1646
Unknown                   285
Name: count, dtype: int64

=== Stage Values ===
STAGE_HIGHEST_RECORDED
Stage 4      3209
Stage 1-3    2901
Unknown         1
Name: count, dtype: int64

=== Cancer Subtypes ===
CANCER_TYPE_DETAILED
Lung Adenocarcinoma                                 4645
Lung Squamous Cell Carcinoma                         641
Non-Small Cell Lung Cancer                           441
Large Cell Neuroendocrine Carcinoma                  111
Poorly Differentiated Non-Small Cell Lung Cancer      67
Lung Adenosquamous Carcinoma                          45
Lung Carcinoid                                        31
Pleomorphic Carcinoma of the Lung                     30
Atypical Lung Carcinoid                               29
Lung Neuroendocrine Tumor                             29
Name: count, dtype: int64

=== TMB Stats 

In [20]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Load data
df = pd.read_csv('nsclc_step1.csv')

# =====================
# Step 1: Remove single unknown stage
# =====================
df = df[df['STAGE_HIGHEST_RECORDED'] != 'Unknown'].copy()
print(f"Patients after stage filter: {len(df)}")

# =====================
# Step 2: Encode STAGE
# =====================
df['STAGE_ENC'] = df['STAGE_HIGHEST_RECORDED'].map({
    'Stage 1-3': 0,
    'Stage 4': 1
})

# =====================
# Step 3: Encode SMOKING
# =====================
df['SMOKING_ENC'] = df['SMOKING_PREDICTIONS_3_CLASSES'].map({
    'Never': 0,
    'Former/Current Smoker': 1,
    'Unknown': np.nan
})

# =====================
# Step 4: Encode CANCER SUBTYPE
# =====================
def simplify_subtype(subtype):
    # Simplify cancer subtypes into main categories
    if 'Adenocarcinoma' in str(subtype):
        return 'Adenocarcinoma'
    elif 'Squamous' in str(subtype):
        return 'Squamous'
    elif 'Neuroendocrine' in str(subtype) or 'Carcinoid' in str(subtype):
        return 'Neuroendocrine'
    else:
        return 'Other_NSCLC'

df['CANCER_SUBTYPE'] = df['CANCER_TYPE_DETAILED'].apply(
    simplify_subtype)

print("\n=== Cancer Subtype Encoded ===")
print(df['CANCER_SUBTYPE'].value_counts())

# One-hot encode cancer subtype
subtype_dummies = pd.get_dummies(
    df['CANCER_SUBTYPE'], prefix='SUBTYPE')
df = pd.concat([df, subtype_dummies], axis=1)

# =====================
# Step 5: Encode TREATMENT
# =====================
df['TREATMENT_ENC'] = df['TREATMENT_FINAL'].map({
    'Chemotherapy': 0,
    'Immunotherapy': 1,
    'Targeted': 2
})

print("\n=== Treatment Encoded ===")
print(df['TREATMENT_FINAL'].value_counts())

# =====================
# Step 6: Normalize TMB (log transform)
# =====================
df['TMB_LOG'] = np.log1p(df['TMB_NONSYNONYMOUS'])

# =====================
# Step 7: Normalize MSI
# =====================
df['MSI_NORM'] = df['MSI_SCORE'].clip(lower=0)

# =====================
# Step 8: Define clinical features list
# =====================
clinical_features = [
    # Demographics
    'GENDER_ENC',
    'CURRENT_AGE_DEID',
    # Clinical
    'STAGE_ENC',
    'SMOKING_ENC',
    # Lab values (cheap tests)
    'TMB_LOG',
    'MSI_NORM',
    'TUMOR_PURITY',
    # Cancer subtype
    'SUBTYPE_Adenocarcinoma',
    'SUBTYPE_Squamous',
    'SUBTYPE_Neuroendocrine',
    'SUBTYPE_Other_NSCLC',
    # Metastasis sites
    'BONE', 'CNS_BRAIN', 'LIVER', 'LUNG', 'LYMPH_NODES'
]

# =====================
# Step 9: Check missing in clinical features
# =====================
print("\n=== Missing in Clinical Features ===")
for col in clinical_features:
    if col in df.columns:
        missing = df[col].isna().sum()
        pct = missing / len(df) * 100
        print(f"{col:35} {missing:4} ({pct:.1f}%)")

# =====================
# Step 10: Check targets
# =====================
print("\n=== Targets ===")
print(f"OS_EVENT distribution:\n{df['OS_EVENT'].value_counts()}")
print(f"\nOS_MONTHS stats:\n{df['OS_MONTHS'].describe()}")
print(f"\nTREATMENT_ENC distribution:\n{df['TREATMENT_ENC'].value_counts()}")

# Save
df.to_csv('nsclc_step2.csv', index=False)
print("\nStep 2 saved!")

Patients after stage filter: 6110

=== Cancer Subtype Encoded ===
CANCER_SUBTYPE
Adenocarcinoma    4647
Squamous           641
Other_NSCLC        622
Neuroendocrine     200
Name: count, dtype: int64

=== Treatment Encoded ===
TREATMENT_FINAL
Immunotherapy    2666
Chemotherapy     2060
Targeted         1384
Name: count, dtype: int64

=== Missing in Clinical Features ===
GENDER_ENC                             0 (0.0%)
CURRENT_AGE_DEID                       0 (0.0%)
STAGE_ENC                              0 (0.0%)
SMOKING_ENC                          284 (4.6%)
TMB_LOG                                0 (0.0%)
MSI_NORM                             177 (2.9%)
TUMOR_PURITY                         302 (4.9%)
SUBTYPE_Adenocarcinoma                 0 (0.0%)
SUBTYPE_Squamous                       0 (0.0%)
SUBTYPE_Neuroendocrine                 0 (0.0%)
SUBTYPE_Other_NSCLC                    0 (0.0%)
BONE                                   0 (0.0%)
CNS_BRAIN                              0 (0.0%)
LIVE

In [21]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

# Load data
df = pd.read_csv('nsclc_step2.csv')

# =====================
# Step 1: Impute missing values
# =====================

# SMOKING: impute with most frequent
df['SMOKING_ENC'] = df['SMOKING_ENC'].fillna(
    df['SMOKING_ENC'].mode()[0])

# MSI_NORM: impute with median
df['MSI_NORM'] = df['MSI_NORM'].fillna(
    df['MSI_NORM'].median())

# TUMOR_PURITY: impute with median
df['TUMOR_PURITY'] = df['TUMOR_PURITY'].fillna(
    df['TUMOR_PURITY'].median())

print("=== After Imputation ===")
key_cols = ['SMOKING_ENC', 'MSI_NORM', 'TUMOR_PURITY']
for col in key_cols:
    print(f"{col}: {df[col].isna().sum()} missing")

# =====================
# Step 2: Encode metastasis sites as binary
# =====================
site_cols = ['BONE', 'CNS_BRAIN', 'LIVER', 
             'LUNG', 'LYMPH_NODES']

for col in site_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})
    df[col] = df[col].fillna(0)

print("\n=== Metastasis Sites Sample ===")
print(df[site_cols].head())

# =====================
# Step 3: Load genomic data (mutations)
# =====================
print("\nLoading mutation data...")
mutations = pd.read_csv('data_mutations.txt', 
                         sep='\t', comment='#',
                         low_memory=False)

print(f"Mutations shape: {mutations.shape}")
print(f"Columns: {mutations.columns.tolist()[:10]}")

# =====================
# Step 4: Find important genes for NSCLC
# =====================
# Key NSCLC genes from literature
nsclc_genes = [
    'EGFR', 'KRAS', 'ALK', 'ROS1', 'BRAF',
    'MET', 'RET', 'NTRK1', 'TP53', 'STK11',
    'KEAP1', 'NF1', 'CDKN2A', 'RB1', 'PTEN',
    'PIK3CA', 'ERBB2', 'FGFR1', 'DDR2', 'NOTCH1'
]

# Count mutations per gene
print("\n=== Top Mutated Genes in Dataset ===")
gene_counts = mutations['Hugo_Symbol'].value_counts()
print(gene_counts.head(20))

# Filter to NSCLC-relevant genes
nsclc_mutations = mutations[
    mutations['Hugo_Symbol'].isin(nsclc_genes)
]
print(f"\nNSCLC-relevant mutations: {len(nsclc_mutations)}")

# Save intermediate
df.to_csv('nsclc_step3_clinical.csv', index=False)
print("\nClinical data saved: nsclc_step3_clinical.csv")

=== After Imputation ===
SMOKING_ENC: 0 missing
MSI_NORM: 0 missing
TUMOR_PURITY: 0 missing

=== Metastasis Sites Sample ===
   BONE  CNS_BRAIN  LIVER  LUNG  LYMPH_NODES
0   0.0        0.0    0.0   1.0          1.0
1   1.0        0.0    1.0   1.0          1.0
2   1.0        0.0    1.0   1.0          1.0
3   1.0        1.0    0.0   1.0          1.0
4   1.0        0.0    1.0   1.0          1.0

Loading mutation data...
Mutations shape: (208544, 123)
Columns: ['Hugo_Symbol', 'Entrez_Gene_Id', 'Center', 'NCBI_Build', 'Chromosome', 'Start_Position', 'End_Position', 'Strand', 'Consequence', 'Variant_Classification']

=== Top Mutated Genes in Dataset ===
Hugo_Symbol
TP53      13876
KRAS       7235
APC        7188
PIK3CA     4178
EGFR       2557
ARID1A     2205
KMT2D      2061
SMAD4      1925
ATM        1681
KMT2C      1656
FAT1       1620
ZFHX3      1571
NF1        1469
CDKN2A     1401
PTPRT      1399
PTEN       1391
PTPRD      1289
FBXW7      1277
STK11      1258
KEAP1      1253
Name: count,

In [23]:
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv('nsclc_step3_clinical.csv')
mutations = pd.read_csv('data_mutations.txt',
                         sep='\t', comment='#',
                         low_memory=False)

# =====================
# Step 1: Filter mutations for NSCLC patients only
# =====================
nsclc_samples = df['SAMPLE_ID'].unique()
mutations_nsclc = mutations[
    mutations['Tumor_Sample_Barcode'].isin(nsclc_samples)
].copy()

print(f"Total mutations in NSCLC patients: {len(mutations_nsclc)}")
print(f"Unique NSCLC samples with mutations: "
      f"{mutations_nsclc['Tumor_Sample_Barcode'].nunique()}")

# =====================
# Step 2: Top mutated genes in NSCLC specifically
# =====================
print("\n=== Top Mutated Genes in NSCLC ===")
nsclc_gene_counts = mutations_nsclc['Hugo_Symbol'].value_counts()
print(nsclc_gene_counts.head(30))

# =====================
# Step 3: Select top 50 genes as genomic features
# =====================
top_genes = nsclc_gene_counts.head(50).index.tolist()
print(f"\nSelected {len(top_genes)} genes as features")

# =====================
# Step 4: Create binary mutation matrix
# one row per sample, one column per gene
# 1 = mutated, 0 = not mutated
# =====================
mutations_filtered = mutations_nsclc[
    mutations_nsclc['Hugo_Symbol'].isin(top_genes)
]

mut_matrix = mutations_filtered.groupby(
    ['Tumor_Sample_Barcode', 'Hugo_Symbol']
).size().unstack(fill_value=0)

# Binarize (1 if mutated, 0 if not)
mut_matrix = (mut_matrix > 0).astype(int)

# Add MUT_ prefix
mut_matrix.columns = ['MUT_' + c for c in mut_matrix.columns]
mut_matrix = mut_matrix.reset_index()
mut_matrix = mut_matrix.rename(
    columns={'Tumor_Sample_Barcode': 'SAMPLE_ID'})

print(f"\nMutation matrix shape: {mut_matrix.shape}")
print(f"Sample of mutation matrix:")
print(mut_matrix.iloc[:3, :8])

# =====================
# Step 5: Merge with clinical data
# =====================
df_final = pd.merge(df, mut_matrix,
                     on='SAMPLE_ID', how='left')

# Fill missing mutations with 0 (gene not mutated)
mut_cols = [c for c in df_final.columns if c.startswith('MUT_')]
df_final[mut_cols] = df_final[mut_cols].fillna(0)

print(f"\nFinal dataset shape: {df_final.shape}")
print(f"Patients with mutation data: "
      f"{df_final[mut_cols[0]].sum()}")

# =====================
# Step 6: Final summary
# =====================
clinical_features = [
    'GENDER_ENC', 'CURRENT_AGE_DEID', 'STAGE_ENC',
    'SMOKING_ENC', 'TMB_LOG', 'MSI_NORM', 'TUMOR_PURITY',
    'SUBTYPE_Adenocarcinoma', 'SUBTYPE_Squamous',
    'SUBTYPE_Neuroendocrine', 'SUBTYPE_Other_NSCLC',
    'BONE', 'CNS_BRAIN', 'LIVER', 'LUNG', 'LYMPH_NODES'
]

genomic_features = mut_cols

print(f"\n=== Final Feature Summary ===")
print(f"Clinical features:  {len(clinical_features)}")
print(f"Genomic features:   {len(genomic_features)}")
print(f"Total features:     "
      f"{len(clinical_features) + len(genomic_features)}")
print(f"\nTarget 1 - OS_EVENT:    {df_final['OS_EVENT'].value_counts().to_dict()}")
print(f"Target 2 - OS_MONTHS:   mean={df_final['OS_MONTHS'].mean():.1f}")
print(f"Target 3 - TREATMENT:   {df_final['TREATMENT_ENC'].value_counts().to_dict()}")

# =====================
# Step 7: Check missing in final dataset
# =====================
print(f"\n=== Missing in Final Dataset ===")
all_features = clinical_features + genomic_features
total_missing = df_final[all_features].isna().sum().sum()
print(f"Total missing values: {total_missing}")

# Save final dataset
df_final.to_csv('nsclc_final.csv', index=False)
print(f"\nFinal dataset saved: nsclc_final.csv")
print(f"Ready for modeling!")

Total mutations in NSCLC patients: 53188
Unique NSCLC samples with mutations: 5866

=== Top Mutated Genes in NSCLC ===
Hugo_Symbol
TP53       3596
EGFR       1857
KRAS       1669
KEAP1       872
STK11       820
RBM10       609
PTPRD       606
NF1         553
SMARCA4     524
PTPRT       513
KMT2D       500
FAT1        499
ARID1A      487
ATM         485
CDKN2A      464
EPHA5       395
PIK3CA      382
SETD2       363
EPHA3       357
RB1         355
KMT2C       340
MET         338
ERBB4       338
ATRX        338
MGA         338
GRIN2A      336
TERT        329
BRAF        323
ZFHX3       311
NTRK3       309
Name: count, dtype: int64

Selected 50 genes as features

Mutation matrix shape: (5682, 51)
Sample of mutation matrix:
           SAMPLE_ID  MUT_ALK  MUT_AMER1  MUT_APC  MUT_ARID1A  MUT_ARID2  \
0  P-0000012-T03-IM3        0          0        0           0          0   
1  P-0000036-T01-IM3        0          0        0           0          0   
2  P-0000082-T01-IM3        0          0  

In [24]:
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv('nsclc_step3_clinical.csv')
mutations = pd.read_csv('data_mutations.txt',
                         sep='\t', comment='#',
                         low_memory=False)

# =====================
# Step 1: Check ID formats
# =====================
print("=== Clinical SAMPLE_ID format ===")
print(df['SAMPLE_ID'].head(5).tolist())

print("\n=== Mutation Tumor_Sample_Barcode format ===")
print(mutations['Tumor_Sample_Barcode'].head(5).tolist())

# =====================
# Step 2: Extract PATIENT_ID from mutation barcode
# P-0000012-T03-IM3 → P-0000012
# =====================
mutations['PATIENT_ID'] = mutations[
    'Tumor_Sample_Barcode'].apply(
    lambda x: '-'.join(str(x).split('-')[:2]))

print("\n=== Extracted PATIENT_ID from mutations ===")
print(mutations['PATIENT_ID'].head(5).tolist())

# =====================
# Step 3: Check overlap
# =====================
nsclc_patients = set(df['PATIENT_ID'].unique())
mutation_patients = set(mutations['PATIENT_ID'].unique())
overlap = nsclc_patients & mutation_patients

print(f"\nNSCLC patients:           {len(nsclc_patients)}")
print(f"Patients with mutations:  {len(mutation_patients)}")
print(f"Overlap:                  {len(overlap)}")

# =====================
# Step 4: Filter mutations for NSCLC patients
# =====================
mutations_nsclc = mutations[
    mutations['PATIENT_ID'].isin(nsclc_patients)
].copy()

print(f"\nNSCLC mutations: {len(mutations_nsclc)}")

# =====================
# Step 5: Top genes in NSCLC
# =====================
print("\n=== Top Mutated Genes in NSCLC ===")
print(mutations_nsclc['Hugo_Symbol'].value_counts().head(20))

# =====================
# Step 6: Select top 50 genes
# =====================
top_genes = mutations_nsclc[
    'Hugo_Symbol'].value_counts().head(50).index.tolist()

# =====================
# Step 7: Create mutation matrix per PATIENT_ID
# =====================
mutations_filtered = mutations_nsclc[
    mutations_nsclc['Hugo_Symbol'].isin(top_genes)
]

mut_matrix = mutations_filtered.groupby(
    ['PATIENT_ID', 'Hugo_Symbol']
).size().unstack(fill_value=0)

mut_matrix = (mut_matrix > 0).astype(int)
mut_matrix.columns = ['MUT_' + c for c in mut_matrix.columns]
mut_matrix = mut_matrix.reset_index()

print(f"\nMutation matrix shape: {mut_matrix.shape}")

# =====================
# Step 8: Merge on PATIENT_ID
# =====================
df_final = pd.merge(df, mut_matrix,
                     on='PATIENT_ID', how='left')

mut_cols = [c for c in df_final.columns 
            if c.startswith('MUT_')]
df_final[mut_cols] = df_final[mut_cols].fillna(0)

# =====================
# Step 9: Verify overlap
# =====================
patients_with_mut = (df_final[mut_cols].sum(axis=1) > 0).sum()
print(f"\nPatients with mutation data: {patients_with_mut}")
print(f"Total patients:              {len(df_final)}")
print(f"Coverage:                    "
      f"{patients_with_mut/len(df_final)*100:.1f}%")

# =====================
# Step 10: Final check
# =====================
print(f"\n=== Final Dataset ===")
print(f"Shape: {df_final.shape}")
print(f"Missing values: {df_final[mut_cols].isna().sum().sum()}")

# Save
df_final.to_csv('nsclc_final.csv', index=False)
print("\nSaved: nsclc_final.csv")

=== Clinical SAMPLE_ID format ===
['P-0000012-T03-IM3', 'P-0000036-T01-IM3', 'P-0000082-T01-IM3', 'P-0000110-T01-IM3', 'P-0000133-T01-IM3']

=== Mutation Tumor_Sample_Barcode format ===
['P-0081657-T01-IM7', 'P-0081657-T01-IM7', 'P-0081657-T01-IM7', 'P-0083825-T01-IM7', 'P-0083825-T01-IM7']

=== Extracted PATIENT_ID from mutations ===
['P-0081657', 'P-0081657', 'P-0081657', 'P-0083825', 'P-0083825']

NSCLC patients:           6110
Patients with mutations:  23785
Overlap:                  5870

NSCLC mutations: 53800

=== Top Mutated Genes in NSCLC ===
Hugo_Symbol
TP53       3622
EGFR       1859
KRAS       1686
KEAP1       872
STK11       820
RBM10       609
PTPRD       609
NF1         554
SMARCA4     530
PTPRT       513
KMT2D       508
FAT1        506
ARID1A      496
ATM         485
CDKN2A      468
EPHA5       397
PIK3CA      396
SETD2       364
EPHA3       359
RB1         356
Name: count, dtype: int64

Mutation matrix shape: (5686, 51)

Patients with mutation data: 5686
Total patients